In [42]:
import sys
!{sys.executable} -m pip install opencv-python
!{sys.executable} -m pip install IMAGEIO
!{sys.executable} -m pip install albumentations
!{sys.executable} -m pip install -q clodsa
!{sys.executable} -m pip install -q git+https://github.com/aleju/imgaug.git
!{sys.executable} -m pip install imagecorruptions

In [45]:
!pip install future
!pip install clodsa


In [2]:
import os
import numpy as np
import cv2 
from glob import glob
from tqdm import tqdm
import imageio
import albumentations as A
from albumentations import *
import sklearn
from sklearn import *
import pandas as pd
from matplotlib import pyplot as plt
import random
import imageio
import imgaug as ia
import imgaug.augmenters as iaa

In [3]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [4]:
def load_data(path):
    # x = images and y = masks
    images_0 = sorted(glob(os.path.join(path, "images_0", "*.tif")))
    masks_0 = sorted(glob(os.path.join(path, "masks_0", "*.gif")))
    images_1 = sorted(glob(os.path.join(path, "images_1", "*.tif")))
    masks_1 = sorted(glob(os.path.join(path, "masks_1", "*.gif")))
    images_2 = sorted(glob(os.path.join(path, "images_2", "*.JPG")))
    masks_2 = sorted(glob(os.path.join(path, "masks_2", "*.tif")))
    images_3 = sorted(glob(os.path.join(path, "images_3", "*.jpg")))
    masks_3 = sorted(glob(os.path.join(path, "masks_3", "*.png")))
    images_4 = sorted(glob(os.path.join(path, "images_4", "*.ppm")))
    masks_4 = sorted(glob(os.path.join(path, "masks_4", "*.ppm")))


    return (images_0, masks_0), (images_1, masks_1), (images_2, masks_2), (images_3, masks_3), (images_4, masks_4)

In [5]:
def process_data(images, mask, save_path, stage, ma_filetype_gif = False):
    IMG_H = 512
    IMG_W = 512
    index = 1
    if stage == 0: 
        index = 1
    elif stage == 1: 
        index = 21
    elif stage == 2:
        index = 41
    elif stage == 3: 
        index = 56
    else:
        index = 84
    for idx, (x, y) in tqdm(enumerate(zip(images, mask)), total= len(images)):
        
        #get image
        
        x = cv2.imread(x, cv2.IMREAD_GRAYSCALE)
   
        if ma_filetype_gif == True:
            y = imageio.mimread(y)[0]

        else:
            y = cv2.imread(y, cv2.IMREAD_GRAYSCALE)
            

        X = [x]
        Y = [y]

        
        for (i, m) in zip(X, Y):
           # i = cv2.resize(i, (IMG_W, IMG_H))
           # m = cv2.resize(m, (IMG_W, IMG_H))
            

            i = cv2.resize(i, (IMG_W, IMG_H))
            m = cv2.resize(m, (IMG_W, IMG_H))
            
            
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            i = clahe.apply(x)
            m = clahe.apply(y)

            
            tmp_image_name = f"{index}.jpg"
            tmp_mask_name = f"{index}.jpg"

            image_path = os.path.join(save_path, "image_total", tmp_image_name)
            mask_path = os.path.join(save_path, "mask_total", tmp_mask_name)

            cv2.imwrite(image_path, i)
            cv2.imwrite(mask_path, m)

            index = index + 1

In [28]:
(images_0, masks_0), (images_1, masks_1), (images_2, masks_2), (images_3, masks_3), (images_4, masks_4) = load_data('/Users/sadda/Documents/Science_Fair/Datasets/RSEG/')
create_dir('/Users/sadda/Documents/Science_Fair/Datasets/RSEG/image_total/')
create_dir('/Users/sadda/Documents/Science_Fair/Datasets/RSEG/mask_total/')

In [29]:
save_path_data = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/'
process_data(images_0, masks_0, save_path_data, stage = 0, ma_filetype_gif = True)
process_data(images_1, masks_1, save_path_data, stage = 1, ma_filetype_gif = True)
process_data(images_2, masks_2, save_path_data, stage = 2)
process_data(images_3, masks_3, save_path_data, stage = 3)
process_data(images_4, masks_4, save_path_data, stage = 4)


100%|███████████████████████████████████████████| 20/20 [00:00<00:00, 98.81it/s]


In [6]:
def load_data_tt(path):
    images = sorted(glob(os.path.join(path, "image_total", "*.jpg")))
    masks = sorted(glob(os.path.join(path, "mask_total", "*.jpg")))
    df = pd.DataFrame(images)
    df[1] = pd.DataFrame(masks)
    train_df, test_df = sklearn.model_selection.train_test_split(df, test_size=0.2, train_size=0.8, random_state=15, shuffle=True)
    train_im = list(train_df[0])
    train_ma = list(train_df[1])
    test_im = list(test_df[0])
    test_ma = list(test_df[1])
    return (train_im, train_ma), (test_im, test_ma)    

In [7]:
def train_test_data(images, mask, save_path):

    IMG_H = 512
    IMG_W = 512
    
    for idx, (x, y) in tqdm(enumerate(zip(images, mask)), total= len(images)):

        name = x.split('/')[-1].split('.')[0]

        #get image
        x = cv2.imread(x, cv2.IMREAD_GRAYSCALE)
        y = cv2.imread(y, cv2.IMREAD_GRAYSCALE)
            


        X = [x]
        Y = [y]

        for i, m in zip(X, Y):
            i = cv2.resize(i, (IMG_W, IMG_H))
            m = cv2.resize(m, (IMG_W, IMG_H))

            tmp_image_name = f"{name}.jpg"
            tmp_mask_name = f"{name}.jpg"


            image_path = os.path.join(save_path, "image", tmp_image_name)
            mask_path = os.path.join(save_path, "mask", tmp_mask_name)

            cv2.imwrite(image_path, i)
            cv2.imwrite(mask_path, m)



In [71]:
(train_im, train_ma), (test_im, test_ma) = load_data_tt('/Users/sadda/Documents/Science_Fair/Datasets/RSEG')
# (train_im, train_ma), (test_im, test_ma)  
train_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/train/'
train_test_data(train_im, train_ma, train_path)
test_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/test/'
train_test_data(test_im, test_ma, test_path)

100%|██████████████████████████████████████████| 21/21 [00:00<00:00, 155.47it/s]


In [8]:
def augment_data(images, mask, save_path, augment=True):
    IMG_H = 512
    IMG_W = 512

    for idx, (x, y) in tqdm(enumerate(zip(images, mask)), total= len(images)):
        # name extraction
        name = x.split('/')[-1].split('.')[0]

        #get image
        x = cv2.imread(x, cv2.IMREAD_GRAYSCALE)
        y = cv2.imread(y, cv2.IMREAD_GRAYSCALE)



        if augment == True:
            aug = HorizontalFlip(p = 1.0)
            augmented = aug(image = x, mask = y)
            x1 = augmented['image']
            y1 = augmented['mask']

            aug = VerticalFlip(p=1.0)
            augmented = aug(image=x, mask=y)
            x2 = augmented['image']
            y2 = augmented['mask']

            aug = GaussianBlur(blur_limit=(3,7), sigma_limit=0, p=1)
            augmented = aug(image=x, mask=y)
            x3 = augmented['image']
            y3 = augmented['mask']

            aug = A.Compose([
            A.OneOf([
            A.GridDistortion(p=0.5),
            A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.5),
            A.OpticalDistortion(distort_limit=2, shift_limit=0.5, p=1)
            ], p=0.8)])
            augmented = aug(image=x, mask=y)
            x4 = augmented['image']
            y4 = augmented['mask']

            aug = A.Compose([
            A.OneOf([
            A.PixelDropout(p=0.02),
            A.RandomGamma(p=1)
            ], p=0.8)])
            augmented = aug(image=x, mask=y)
            x5 = augmented['image']
            y5 = augmented['mask']
            
            aug = RandomSizedCrop(min_max_height=(128, 448), height=IMG_H, width=IMG_W, p=1)
            augmented = aug(image = x, mask = y)
            x6 = augmented['image']
            y6 = augmented['mask']
            
            aug = Rotate(limit=75, interpolation=1, border_mode=4, rotate_method='largest_box', p=1)
            augmented = aug(image = x, mask = y)
            x7 = augmented['image']
            y7 = augmented['mask']
            
            aug = Transpose(p=1)
            augmented = aug(image = x, mask = y)
            x8= augmented['image']
            y8= augmented['mask']

            aug = A.Compose([
            A.OneOf([
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.5),
            A.RandomBrightnessContrast(p = 1.0)
            ], p=0.8)])
            augmented = aug(image=x, mask=y)
            x9 = augmented['image']
            y9 = augmented['mask']
            
            aug = JpegCompression(quality_lower=96, quality_upper=100, p=1)
            augmented = aug(image = x, mask = y)
            x10= augmented['image']
            y10= augmented['mask']

            
            
            

            X = [x, x1, x2, x3, x4, x5, x6, x7, x8, x9, x10]
            Y = [y, y1, y2, y3, y4, y5, y6, y7, y8, y9, y10]


        index = 0
        for i, m in zip(X, Y):
            i = cv2.resize(i, (IMG_W, IMG_H))
            m = cv2.resize(m, (IMG_W, IMG_H))


            tmp_image_name = f"{name}_{index}.jpg"
            tmp_mask_name = f"{name}_{index}.jpg"

            image_path = os.path.join(save_path, "image", tmp_image_name)
            mask_path = os.path.join(save_path, "mask", tmp_mask_name)

            cv2.imwrite(image_path, i)
            cv2.imwrite(mask_path, m)

            index = index + 1


In [9]:
path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/train/'
save_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/aug_train/'
create_dir(os.path.join(save_path, 'image'))
create_dir(os.path.join(save_path, 'mask'))

def load_data_aug(path):
    images = sorted(glob(os.path.join(path, "images", "*.jpg")))
    masks = sorted(glob(os.path.join(path, "labels", "*.jpg")))
    return (images, masks)
(images, masks) = load_data_aug(path)
augment_data(images, masks, save_path, augment=True)

  0%|                                                    | 0/82 [00:00<?, ?it/s]/Users/sadda/miniconda3/envs/tensorflow/lib/python3.10/site-packages/albumentations/augmentations/transforms.py:316: FutureWarning: JpegCompression has been deprecated. Please use ImageCompression
  warnings.warn(
100%|███████████████████████████████████████████| 82/82 [00:03<00:00, 26.18it/s]


In [10]:
## new imports

import skimage.io as io
import skimage.transform as trans
import tensorflow as tf
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras import backend as keras
import matplotlib.pyplot as plt
import scipy.misc as sc
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, ReduceLROnPlateau, EarlyStopping, TensorBoard
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.metrics import Recall, Precision, Accuracy



In [11]:

def adjustData(img,mask,flag_multi_class,num_class):
    if(flag_multi_class):
        img = img/255
        mask = mask[:,:,:,0] if(len(mask.shape) == 4) else mask[:,:,0]
        new_mask = np.zeros(mask.shape + (num_class,))
        for i in range(num_class):
            #for one pixel in the image, find the class in mask and convert it into one-hot vector
            #index = np.where(mask == i)
            #index_mask = (index[0],index[1],index[2],np.zeros(len(index[0]),dtype = np.int64) + i) if (len(mask.shape) == 4) else (index[0],index[1],np.zeros(len(index[0]),dtype = np.int64) + i)
            #new_mask[index_mask] = 1
            new_mask[mask == i,i] = 1
        new_mask = np.reshape(new_mask,(new_mask.shape[0],new_mask.shape[1]*new_mask.shape[2],new_mask.shape[3])) if flag_multi_class else np.reshape(new_mask,(new_mask.shape[0]*new_mask.shape[1],new_mask.shape[2]))
        mask = new_mask
    elif(np.max(img) > 1):
        img = img / 255
        mask = mask /255
        mask[mask > 0.5] = 1
        mask[mask <= 0.5] = 0
        #print(np.shape(mask),np.shape(img))
    return (img,mask)



def trainGenerator(batch_size,train_path,image_folder,mask_folder,aug_dict,image_color_mode = "grayscale",
                    mask_color_mode = "grayscale",image_save_prefix  = "image",mask_save_prefix  = "mask",
                    flag_multi_class = False,num_class = 2,save_to_dir = None,target_size = (512,512),seed = 2):
    '''
    can generate image and mask at the same time
    use the same seed for image_datagen and mask_datagen to ensure the transformation for image and mask is the same
    if you want to visualize the results of generator, set save_to_dir = "your path"
    '''
    image_datagen = ImageDataGenerator(**aug_dict)
    mask_datagen = ImageDataGenerator(**aug_dict)
    image_generator = image_datagen.flow_from_directory(
        train_path,
        classes = [image_folder],
        class_mode = None,
        color_mode = image_color_mode,
        target_size = target_size,
        batch_size = batch_size,
        save_to_dir = save_to_dir,
        save_prefix  = image_save_prefix,
        seed = seed)
    mask_generator = mask_datagen.flow_from_directory(
        train_path,
        classes = [mask_folder],
        class_mode = None,
        color_mode = mask_color_mode,
        target_size = target_size,
        batch_size = batch_size,
        save_to_dir = save_to_dir,
        
        save_prefix  = mask_save_prefix,
        seed = seed)
    train_generator = zip(image_generator, mask_generator)
    for (img,mask) in train_generator:
        img,mask = adjustData(img,mask,flag_multi_class,num_class)
        yield (img,mask)


def testGenerator(test_path,num_image = 30,target_size = (512,512),flag_multi_class = False,as_gray = True):
    image_datagen = ImageDataGenerator(rescale=1./255)
    mask_datagen = ImageDataGenerator(rescale=1./255)
    files=sorted(os.listdir(test_path))
    num_image=len(files)
    for i in range(num_image):
        img = io.imread(os.path.join(test_path,files[i]),as_gray = as_gray)
        #img = cv2.imread(os.path.join(test_path,files[i]), cv2.IMREAD_GRAYSCALE)
        print(files[i])
        img = trans.resize(img,target_size)
        img = np.reshape(img,img.shape+(1,)) if (not flag_multi_class) else img
        img = np.reshape(img,(1,)+img.shape)
        yield img
        


def validGenerator(batch_size,test_path,image_folder,mask_folder,aug_dict,image_color_mode = "grayscale",
                    mask_color_mode = "grayscale",image_save_prefix  = "image",mask_save_prefix  = "mask",
                    flag_multi_class = False,num_class = 2,save_to_dir = None,target_size = (512,512),seed = 1):
    image_datagen = ImageDataGenerator(**aug_dict)
    mask_datagen = ImageDataGenerator(**aug_dict)
    image_generator = image_datagen.flow_from_directory(
      test_path,
      classes = [image_folder],
      class_mode = None,
      color_mode = image_color_mode,
      target_size = target_size,
      batch_size = batch_size,
      save_to_dir = save_to_dir,
      save_prefix  = image_save_prefix,
      seed = seed)
    mask_generator = mask_datagen.flow_from_directory(
      test_path,
      classes = [mask_folder],
      class_mode = None,
      color_mode = mask_color_mode,
      target_size = target_size,
      batch_size = batch_size,
      save_to_dir = save_to_dir,
      save_prefix  = mask_save_prefix,
      seed = seed)
    valid_generator = zip(image_generator, mask_generator)
    for(img,mask) in valid_generator:
        img,mask = adjustData(img,mask,flag_multi_class,num_class)
        yield (img,mask)

In [12]:
def labelVisualize(num_class,color_dict,img): # save test images
    img = img[:,:,0] if len(img.shape) == 3 else img
    img_out = np.zeros(img.shape + (3,))
    for i in range(num_class):
        img_out[img == i] = color_dict[i]
      
    return img_out


def saveResult(img_path,save_path,npyfile,flag_multi_class = False,num_class = 2):
    files=os.listdir(img_path)
    #print(len(img_path))
    #print(len(npyfile))
    
    for i,item in enumerate(npyfile):
        img = labelVisualize(num_class,COLOR_DICT,item) if flag_multi_class else item[:,:,0]
        #img1=np.array(((img - np.min(img))/np.ptp(img))>0.6).astype(float)
        img[img>0.1]=1
        img[img<=0.1]=0
        io.imsave(os.path.join(save_path, files[i]+'_predict.png'),img)
        


In [13]:
#metrics
from sklearn.metrics import accuracy_score, f1_score, jaccard_score, precision_score, recall_score

def mean_iou(true_y, pred_y):#iou
    def f(true_y, pred_y):
        intersection = (true_y*pred_y).sum()
        union = true_y.sum() + pred_y.sum() - intersection
        x = (intersection + 1e-15) / (union + 1e-15)
        x = x.astype(np.float32)
        return x
    return tf.numpy_function(f, [true_y, pred_y], tf.float32)

smooth = 1e-15
def dice_coef(true_y, pred_y):
    true_y = tf.keras.layers.Flatten()(true_y)
    pred_y = tf.keras.layers.Flatten()(pred_y)
    intersection = tf.reduce_sum(true_y * pred_y)
    return (2. * intersection + smooth) / (tf.reduce_sum(true_y) + tf.reduce_sum(pred_y) + smooth)

def dice_loss(true_y, pred_y):
    return 1.0 - dice_coef(true_y, pred_y)


def dice(true_mask, pred_mask, non_seg_score=1.0):
    assert true_mask.shape == pred_mask.shape

    true_mask = np.asarray(true_mask).astype(np.bool)
    pred_mask = np.asarray(pred_mask).astype(np.bool)

    # If both segmentations are all zero, the dice will be 1. (Developer decision)
    #im_sum = true_mask.sum() + pred_mask.sum()
    #if im_sum == 0:
    #    return non_seg_score

    # Compute Dice coefficient
    intersection = np.logical_and(true_mask, pred_mask)
    return 2. * intersection.sum() / im_sum
     

def mean_dice(true_path, pred_path):
  
  sum = 0
  
  for img in sorted(os.listdir(pred_path)):
    
    true_tmp = cv2.imread(true_path + '/' + img, 0)
    pred_tmp = cv2.imread(pred_path + '/' + img, 0)
    
    a = dice(true_tmp, pred_tmp)
    print(a)
    sum += a
  
  return sum/len(os.listdir(true_path))

In [14]:
data_gen_args = dict() 

In [15]:
import tensorflow as tf
from tensorflow.keras import models, layers, regularizers
from tensorflow.keras import backend as K

def repeat_elem(tensor, rep):
    # lambda function to repeat Repeats the elements of a tensor along an axis
    #by a factor of rep.
    # If tensor has shape (None, 256,256,3), lambda will return a tensor of shape 
    #(None, 256,256,6), if specified axis=3 and rep=2.

     return layers.Lambda(lambda x, repnum: K.repeat_elements(x, repnum, axis=3),
                          arguments={'repnum': rep})(tensor)

def gating_signal(input, out_size, batch_norm=False):
    """
    resize the down layer feature map into the same dimension as the up layer feature map
    using 1x1 conv
    :return: the gating feature map with the same dimension of the up layer feature map
    """
    x = layers.Conv2D(out_size, (1, 1), padding='same')(input)
    if batch_norm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def attention_block(x, gating, inter_shape):
    shape_x = K.int_shape(x)
    shape_g = K.int_shape(gating)

# Getting the x signal to the same shape as the gating signal
    theta_x = layers.Conv2D(inter_shape, (2, 2), strides=(2, 2), padding='same')(x)  # 16
    shape_theta_x = K.int_shape(theta_x)

# Getting the gating signal to the same number of filters as the inter_shape
    phi_g = layers.Conv2D(inter_shape, (1, 1), padding='same')(gating)
    upsample_g = layers.Conv2DTranspose(inter_shape, (3, 3),
                                 strides=(shape_theta_x[1] // shape_g[1], shape_theta_x[2] // shape_g[2]),
                                 padding='same')(phi_g)  # 16

    concat_xg = layers.add([upsample_g, theta_x])
    act_xg = layers.Activation('relu')(concat_xg)
    psi = layers.Conv2D(1, (1, 1), padding='same')(act_xg)
    sigmoid_xg = layers.Activation('sigmoid')(psi)
    shape_sigmoid = K.int_shape(sigmoid_xg)
    upsample_psi = layers.UpSampling2D(size=(shape_x[1] // shape_sigmoid[1], shape_x[2] // shape_sigmoid[2]))(sigmoid_xg)  # 32

    upsample_psi = repeat_elem(upsample_psi, shape_x[3])

    y = layers.multiply([upsample_psi, x])

    result = layers.Conv2D(shape_x[3], (1, 1), padding='same')(y)
    result_bn = layers.BatchNormalization()(result)
    return result_bn

In [16]:
'''def res_conv_block(x, filter_size, size, dropout, batch_norm=False):# also rec block
    
    Residual convolutional layer.
    Two variants....
    Either put activation function before the addition with shortcut
    or after the addition (which would be as proposed in the original resNet).
    
    1. conv - BN - Activation - conv - BN - Activation 
                                          - shortcut  - BN - shortcut+BN
                                          
    2. conv - BN - Activation - conv - BN   
                                     - shortcut  - BN - shortcut+BN - Activation                                     
    
    Check fig 4 in https://arxiv.org/ftp/arxiv/papers/1802/1802.06955.pdf
    
    shortcut = layers.Conv2D(size, kernel_size=(1, 1), padding='same')(x)
    if batch_norm is True:
        shortcut = layers.BatchNormalization(axis=3)(shortcut)
        
    layer = shortcut
    for j in range(2):

        for i in range(2):
            if i == 0:

                conv = layers.Conv2D(size, (filter_size, filter_size), padding='same')(layer)
                if batch_norm is True:
                    conv = layers.BatchNormalization(axis=3)(conv)
                conv = layers.Activation('relu')(conv)
            conv = layers.Conv2D(size, (filter_size, filter_size), padding='same')(add([conv, layer]))
            if batch_norm is True:
                conv = layers.BatchNormalization(axis=3)(conv)
            conv = layers.Activation('relu')(conv)
        layer = conv
        
    if dropout > 0:
        layer = layers.Dropout(dropout)(layer)

    res_path = layers.add([shortcut, layer])
    res_path = layers.Activation('relu')(res_path)    #Activation after addition with shortcut (Original residual block)
    return res_path
    
'''
    

"def res_conv_block(x, filter_size, size, dropout, batch_norm=False):# also rec block\n    \n    Residual convolutional layer.\n    Two variants....\n    Either put activation function before the addition with shortcut\n    or after the addition (which would be as proposed in the original resNet).\n    \n    1. conv - BN - Activation - conv - BN - Activation \n                                          - shortcut  - BN - shortcut+BN\n                                          \n    2. conv - BN - Activation - conv - BN   \n                                     - shortcut  - BN - shortcut+BN - Activation                                     \n    \n    Check fig 4 in https://arxiv.org/ftp/arxiv/papers/1802/1802.06955.pdf\n    \n    shortcut = layers.Conv2D(size, kernel_size=(1, 1), padding='same')(x)\n    if batch_norm is True:\n        shortcut = layers.BatchNormalization(axis=3)(shortcut)\n        \n    layer = shortcut\n    for j in range(2):\n\n        for i in range(2):\n            if

In [17]:
    
def conv_block(x, filter_size, size, dropout, batch_norm=False):
    
    conv = layers.Conv2D(size, (filter_size, filter_size), padding="same")(x)
    if batch_norm is True:
        conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation("relu")(conv)

    conv = layers.Conv2D(size, (filter_size, filter_size), padding="same")(conv)
    if batch_norm is True:
        conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation("relu")(conv)
    
    if dropout > 0:
        conv = layers.Dropout(dropout)(conv)

    return conv
    
    
def res_conv_block(x, filter_size, size, dropout, batch_norm=False):
    '''
    Residual convolutional layer.
    Two variants....
    Either put activation function before the addition with shortcut
    or after the addition (which would be as proposed in the original resNet).
    
    1. conv - BN - Activation - conv - BN - Activation 
                                          - shortcut  - BN - shortcut+BN
                                          
    2. conv - BN - Activation - conv - BN   
                                     - shortcut  - BN - shortcut+BN - Activation                                     
    
    Check fig 4 in https://arxiv.org/ftp/arxiv/papers/1802/1802.06955.pdf

    '''
    conv = layers.Conv2D(size, (filter_size, filter_size), padding='same')(x)
    if batch_norm is True:
        conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation('relu')(conv)
    
    conv = layers.Conv2D(size, (filter_size, filter_size), padding='same')(conv)
    if batch_norm is True:
        conv = layers.BatchNormalization(axis=3)(conv)
    #conv = layers.Activation('relu')(conv)    #Activation before addition with shortcut
    if dropout > 0:
        conv = layers.Dropout(dropout)(conv)

    shortcut = layers.Conv2D(size, kernel_size=(1, 1), padding='same')(x)
    if batch_norm is True:
        shortcut = layers.BatchNormalization(axis=3)(shortcut)

    res_path = layers.add([shortcut, conv])
    res_path = layers.Activation('relu')(res_path)    #Activation after addition with shortcut (Original residual block)
    return res_path


In [18]:
# final model

def Attention_ResUNet(input_shape, pretrained_weights = None, NUM_CLASSES=1, dropout_rate=0.0, batch_norm=True):
    '''
    Residual UNet, with attention 
    
    '''
    # network structure
    FILTER_NUM = 64 # number of basic filters for the first layer
    FILTER_SIZE = 3 # size of the convolutional filter
    UP_SAMP_SIZE = 2 # size of upsampling filters
    # input data
    # dimension of the image depth
    inputs = layers.Input(input_shape, dtype=tf.float32)
    axis = 3

    # Downsampling layers
    # DownRes 1, double residual convolution + pooling
    conv_128 = res_conv_block(inputs, FILTER_SIZE, FILTER_NUM, dropout_rate, batch_norm)
    pool_64 = layers.MaxPooling2D(pool_size=(2,2))(conv_128)
    # DownRes 2
    conv_64 = res_conv_block(pool_64, FILTER_SIZE, 2*FILTER_NUM, dropout_rate, batch_norm)
    pool_32 = layers.MaxPooling2D(pool_size=(2,2))(conv_64)
    # DownRes 3
    conv_32 = res_conv_block(pool_32, FILTER_SIZE, 4*FILTER_NUM, dropout_rate, batch_norm)
    pool_16 = layers.MaxPooling2D(pool_size=(2,2))(conv_32)
    # DownRes 4
    conv_16 = res_conv_block(pool_16, FILTER_SIZE, 8*FILTER_NUM, dropout_rate, batch_norm)
    pool_8 = layers.MaxPooling2D(pool_size=(2,2))(conv_16)
    # DownRes 5, convolution only
    conv_8 = res_conv_block(pool_8, FILTER_SIZE, 16*FILTER_NUM, dropout_rate, batch_norm)
    # Upsampling layers
    # UpRes 6, attention gated concatenation + upsampling + double residual convolution
    gating_16 = gating_signal(conv_8, 8*FILTER_NUM, batch_norm)
    att_16 = attention_block(conv_16, gating_16, 8*FILTER_NUM)
    #drop_8 = Dropout(0.5)(conv_8, training=True)
    up_16 = layers.UpSampling2D(size=(UP_SAMP_SIZE, UP_SAMP_SIZE), data_format="channels_last")(conv_8)
    up_16 = layers.concatenate([up_16, att_16], axis=axis)
    up_conv_16 = res_conv_block(up_16, FILTER_SIZE, 8*FILTER_NUM, dropout_rate, batch_norm)
    # UpRes 7
    gating_32 = gating_signal(up_conv_16, 4*FILTER_NUM, batch_norm)
    att_32 = attention_block(conv_32, gating_32, 4*FILTER_NUM)
    up_32 = layers.UpSampling2D(size=(UP_SAMP_SIZE, UP_SAMP_SIZE), data_format="channels_last")(up_conv_16)
    up_32 = layers.concatenate([up_32, att_32], axis=axis)
    up_conv_32 = res_conv_block(up_32, FILTER_SIZE, 4*FILTER_NUM, dropout_rate, batch_norm)
    # UpRes 8
    gating_64 = gating_signal(up_conv_32, 2*FILTER_NUM, batch_norm)
    att_64 = attention_block(conv_64, gating_64, 2*FILTER_NUM)
    up_64 = layers.UpSampling2D(size=(UP_SAMP_SIZE, UP_SAMP_SIZE), data_format="channels_last")(up_conv_32)
    up_64 = layers.concatenate([up_64, att_64], axis=axis)
    up_conv_64 = res_conv_block(up_64, FILTER_SIZE, 2*FILTER_NUM, dropout_rate, batch_norm)
    # UpRes 9
    gating_128 = gating_signal(up_conv_64, FILTER_NUM, batch_norm)
    att_128 = attention_block(conv_128, gating_128, FILTER_NUM)
    up_128 = layers.UpSampling2D(size=(UP_SAMP_SIZE, UP_SAMP_SIZE), data_format="channels_last")(up_conv_64)
    up_128 = layers.concatenate([up_128, att_128], axis=axis)
    up_conv_128 = res_conv_block(up_128, FILTER_SIZE, FILTER_NUM, dropout_rate, batch_norm)

    # 1*1 convolutional layers
    
    conv_final = layers.Conv2D(NUM_CLASSES, kernel_size=(1,1))(up_conv_128)
    conv_final = layers.BatchNormalization(axis=axis)(conv_final)
    conv_final = layers.Activation('sigmoid')(conv_final)  #Change to softmax for multichannel

    # Model integration
    model = models.Model(inputs, conv_final, name="AttentionResUNet")
    model.compile(optimizer = Adam(learning_rate = 1e-4), loss = 'binary_crossentropy', metrics = [dice_coef, mean_iou, Recall(), Precision(), 'accuracy'])
    

    if(pretrained_weights):
        model=keras.models.load_model(pretrained_weights)
    return model




In [19]:
## attention

def Attention_UNet(input_shape, NUM_CLASSES=1, dropout_rate=0.0, batch_norm=True):
    '''
    Attention UNet, 
    
    '''
    # network structure
    FILTER_NUM = 64 # number of basic filters for the first layer
    FILTER_SIZE = 3 # size of the convolutional filter
    UP_SAMP_SIZE = 2 # size of upsampling filters
    
    inputs = layers.Input(input_shape, dtype=tf.float32)

    # Downsampling layers
    # DownRes 1, convolution + pooling
    conv_128 = conv_block(inputs, FILTER_SIZE, FILTER_NUM, dropout_rate, batch_norm)
    pool_64 = layers.MaxPooling2D(pool_size=(2,2))(conv_128)
    # DownRes 2
    conv_64 = conv_block(pool_64, FILTER_SIZE, 2*FILTER_NUM, dropout_rate, batch_norm)
    pool_32 = layers.MaxPooling2D(pool_size=(2,2))(conv_64)
    # DownRes 3
    conv_32 = conv_block(pool_32, FILTER_SIZE, 4*FILTER_NUM, dropout_rate, batch_norm)
    pool_16 = layers.MaxPooling2D(pool_size=(2,2))(conv_32)
    # DownRes 4
    conv_16 = conv_block(pool_16, FILTER_SIZE, 8*FILTER_NUM, dropout_rate, batch_norm)
    pool_8 = layers.MaxPooling2D(pool_size=(2,2))(conv_16)
    # DownRes 5, convolution only
    conv_8 = conv_block(pool_8, FILTER_SIZE, 16*FILTER_NUM, dropout_rate, batch_norm)

    # Upsampling layers
    # UpRes 6, attention gated concatenation + upsampling + double residual convolution
    gating_16 = gating_signal(conv_8, 8*FILTER_NUM, batch_norm)
    att_16 = attention_block(conv_16, gating_16, 8*FILTER_NUM)
    up_16 = layers.UpSampling2D(size=(UP_SAMP_SIZE, UP_SAMP_SIZE), data_format="channels_last")(conv_8)
    up_16 = layers.concatenate([up_16, att_16], axis=3)
    up_conv_16 = conv_block(up_16, FILTER_SIZE, 8*FILTER_NUM, dropout_rate, batch_norm)
    # UpRes 7
    gating_32 = gating_signal(up_conv_16, 4*FILTER_NUM, batch_norm)
    att_32 = attention_block(conv_32, gating_32, 4*FILTER_NUM)
    up_32 = layers.UpSampling2D(size=(UP_SAMP_SIZE, UP_SAMP_SIZE), data_format="channels_last")(up_conv_16)
    up_32 = layers.concatenate([up_32, att_32], axis=3)
    up_conv_32 = conv_block(up_32, FILTER_SIZE, 4*FILTER_NUM, dropout_rate, batch_norm)
    # UpRes 8
    gating_64 = gating_signal(up_conv_32, 2*FILTER_NUM, batch_norm)
    att_64 = attention_block(conv_64, gating_64, 2*FILTER_NUM)
    up_64 = layers.UpSampling2D(size=(UP_SAMP_SIZE, UP_SAMP_SIZE), data_format="channels_last")(up_conv_32)
    up_64 = layers.concatenate([up_64, att_64], axis=3)
    up_conv_64 = conv_block(up_64, FILTER_SIZE, 2*FILTER_NUM, dropout_rate, batch_norm)
    # UpRes 9
    gating_128 = gating_signal(up_conv_64, FILTER_NUM, batch_norm)
    att_128 = attention_block(conv_128, gating_128, FILTER_NUM)
    up_128 = layers.UpSampling2D(size=(UP_SAMP_SIZE, UP_SAMP_SIZE), data_format="channels_last")(up_conv_64)
    up_128 = layers.concatenate([up_128, att_128], axis=3)
    up_conv_128 = conv_block(up_128, FILTER_SIZE, FILTER_NUM, dropout_rate, batch_norm)

    # 1*1 convolutional layers
    conv_final = layers.Conv2D(NUM_CLASSES, kernel_size=(1,1))(up_conv_128)
    conv_final = layers.BatchNormalization(axis=3)(conv_final)
    conv_final = layers.Activation('sigmoid')(conv_final)  #Change to softmax for multichannel

    # Model integration
    model = models.Model(inputs, conv_final, name="Attention_UNet")
    model.compile(optimizer = Adam(learning_rate = 1e-4), loss = 'binary_crossentropy', metrics = [dice_coef, mean_iou, Recall(), Precision(), 'accuracy'])
    

    if(pretrained_weights):
        model=keras.models.load_model(pretrained_weights)
    return model

In [103]:
#from tensorflow.keras.callbacks import TensorBoard

#import datetime
#%load_ext tensorboard
#log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
#tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

#fit

#os.environ['TENSORBOARD_BINARY'] = '/path/to/envs/my_env/bin/tensorboard'

#Visualize on tensorboard
#%tensorboard --logdir logs/fit


The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
300/300 [==============================] - ETA: 0s - loss: 0.2557 - accuracy: 0.9197
Epoch 1: loss improved from inf to 0.25565, saving model to mock.hdf5
300/300 [==============================] - 33s 110ms/step - loss: 0.2557 - accuracy: 0.9197 - val_loss: 0.2440 - val_accuracy: 0.9209 - lr: 1.0000e-04


ERROR: Could not find '/path/to/envs/my_env/bin/tensorboard' (set by
the `TENSORBOARD_BINARY` environment variable). Please ensure that
your PATH contains an executable `tensorboard` program, or explicitly
specify the path to a TensorBoard binary by setting the
`TENSORBOARD_BINARY` environment variable.

In [20]:
#convolutional block
def conv_block(x, kernelsize, filters, dropout, batchnorm=False): 
    conv = layers.Conv2D(filters, (kernelsize, kernelsize), kernel_initializer='he_normal', padding="same")(x)
    if batchnorm is True:
        conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation("relu")(conv)
    if dropout > 0:
        conv = layers.Dropout(dropout)(conv)
    conv = layers.Conv2D(filters, (kernelsize, kernelsize), kernel_initializer='he_normal', padding="same")(conv)
    if batchnorm is True:
        conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation("relu")(conv)
    return conv


#residual convolutional block
def res_conv_block(x, kernelsize, filters, dropout, batchnorm=False):
    conv1 = layers.Conv2D(filters, (kernelsize, kernelsize), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm is True:
        conv1 = layers.BatchNormalization(axis=3)(conv1)
    conv1 = layers.Activation('relu')(conv1)    
    conv2 = layers.Conv2D(filters, (kernelsize, kernelsize), kernel_initializer='he_normal', padding='same')(conv1)
    if batchnorm is True:
        conv2 = layers.BatchNormalization(axis=3)(conv2)
        conv2 = layers.Activation("relu")(conv2)
    if dropout > 0:
        conv2 = layers.Dropout(dropout)(conv2)
        
    #skip connection    
    shortcut = layers.Conv2D(filters, kernel_size=(1, 1), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm is True:
        shortcut = layers.BatchNormalization(axis=3)(shortcut)
    shortcut = layers.Activation("relu")(shortcut)
    respath = layers.add([shortcut, conv2])       
    return respath


#gating signal for attention unit
def gatingsignal(input, out_size, batchnorm=False):
    x = layers.Conv2D(out_size, (1, 1), padding='same')(input)
    if batchnorm:
        x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

#attention unit/block based on soft attention
def attention_block(x, gating, inter_shape):
    shape_x = K.int_shape(x)
    shape_g = K.int_shape(gating)
    theta_x = layers.Conv2D(inter_shape, (2, 2), strides=(2, 2), kernel_initializer='he_normal', padding='same')(x) 
    shape_theta_x = K.int_shape(theta_x)
    phi_g = layers.Conv2D(inter_shape, (1, 1), kernel_initializer='he_normal', padding='same')(gating)
    upsample_g = layers.Conv2DTranspose(inter_shape, (3, 3), strides=(shape_theta_x[1] // shape_g[1], shape_theta_x[2] // shape_g[2]), kernel_initializer='he_normal', padding='same')(phi_g)
    concat_xg = layers.add([upsample_g, theta_x])
    act_xg = layers.Activation('relu')(concat_xg)
    psi = layers.Conv2D(1, (1, 1), kernel_initializer='he_normal', padding='same')(act_xg)
    sigmoid_xg = layers.Activation('sigmoid')(psi)
    shape_sigmoid = K.int_shape(sigmoid_xg)
    upsample_psi = layers.UpSampling2D(size=(shape_x[1] // shape_sigmoid[1], shape_x[2] // shape_sigmoid[2]))(sigmoid_xg) 
    upsample_psi = layers.Lambda(lambda x, repnum: K.repeat_elements(x, repnum, axis=3), arguments={'repnum': shape_x[3]})(upsample_psi)                          
    y = layers.multiply([upsample_psi, x])
    result = layers.Conv2D(shape_x[3], (1, 1), kernel_initializer='he_normal', padding='same')(y)
    attenblock = layers.BatchNormalization()(result)
    return attenblock


    
#Attention U-NET
def attentionunet(input_shape, dropout=0.2, batchnorm=True):
    
    filters = [16, 32, 64, 128, 256]
    kernelsize = 3
    upsample_size = 2

    inputs = layers.Input(input_shape) 

    # Downsampling layers    
    dn_1 = conv_block(inputs, kernelsize, filters[0], dropout, batchnorm)
    pool_1 = layers.MaxPooling2D(pool_size=(2,2))(dn_1)
    
    dn_2 = conv_block(pool_1, kernelsize, filters[1], dropout, batchnorm)
    pool_2 = layers.MaxPooling2D(pool_size=(2,2))(dn_2)
    
    dn_3 = conv_block(pool_2, kernelsize, filters[2], dropout, batchnorm)
    pool_3 = layers.MaxPooling2D(pool_size=(2,2))(dn_3)
    
    dn_4 = conv_block(pool_3, kernelsize, filters[3], dropout, batchnorm)
    pool_4 = layers.MaxPooling2D(pool_size=(2,2))(dn_4)
    
    dn_5 = conv_block(pool_4, kernelsize, filters[4], dropout, batchnorm)

    # Upsampling layers    
    gating_5 = gatingsignal(dn_5, filters[3], batchnorm)
    att_5 = attention_block(dn_4, gating_5, filters[3])
    up_5 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(dn_5)
    up_5 = layers.concatenate([up_5, att_5], axis=3)
    up_conv_5 = conv_block(up_5, kernelsize, filters[3], dropout, batchnorm)
    
    gating_4 = gatingsignal(up_conv_5, filters[2], batchnorm)
    att_4 = attention_block(dn_3, gating_4, filters[2])
    up_4 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_5)
    up_4 = layers.concatenate([up_4, att_4], axis=3)
    up_conv_4 = conv_block(up_4, kernelsize, filters[2], dropout, batchnorm)
   
    gating_3 = gatingsignal(up_conv_4, filters[1], batchnorm)
    att_3 = attention_block(dn_2, gating_3, filters[1])
    up_3 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_4)
    up_3 = layers.concatenate([up_3, att_3], axis=3)
    up_conv_3 = conv_block(up_3, kernelsize, filters[1], dropout, batchnorm)
    
    gating_2 = gatingsignal(up_conv_3, filters[0], batchnorm)
    att_2 = attention_block(dn_1, gating_2, filters[0])
    up_2 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_3)
    up_2 = layers.concatenate([up_2, att_2], axis=3)
    up_conv_2 = conv_block(up_2, kernelsize, filters[0], dropout, batchnorm)
    
    conv_final = layers.Conv2D(1, kernel_size=(1,1))(up_conv_2)
    conv_final = layers.BatchNormalization(axis=3)(conv_final)
    outputs = layers.Activation('sigmoid')(conv_final)  

    model = models.Model(inputs=[inputs], outputs=[outputs])
    model.compile(optimizer = Adam(learning_rate = 1e-4), loss = 'binary_crossentropy', metrics = [dice_coef, mean_iou, Recall(), Precision(), 'accuracy'])


    model.summary()       
    return model    



#Residual-Attention UNET (RA-UNET)
def residual_attentionunet(input_shape, dropout=0.2, batchnorm=True):
    
    filters = [16, 32, 64, 128, 256]
    kernelsize = 3
    upsample_size = 2
    
    inputs = layers.Input(input_shape)    
    
    # Downsampling layers
    dn_1 = res_conv_block(inputs, kernelsize, filters[0], dropout, batchnorm)
    pool1 = layers.MaxPooling2D(pool_size=(2,2))(dn_1)

    dn_2 = res_conv_block(pool1, kernelsize, filters[1], dropout, batchnorm)
    pool2 = layers.MaxPooling2D(pool_size=(2,2))(dn_2)

    dn_3 = res_conv_block(pool2, kernelsize, filters[2], dropout, batchnorm)
    pool3 = layers.MaxPooling2D(pool_size=(2,2))(dn_3)

    dn_4 = res_conv_block(pool3, kernelsize, filters[3], dropout, batchnorm)
    pool4 = layers.MaxPooling2D(pool_size=(2,2))(dn_4)

    dn_5 = res_conv_block(pool4, kernelsize, filters[4], dropout, batchnorm)

    # Upsampling layers    
    gating_5 = gatingsignal(dn_5, filters[3], batchnorm)
    att_5 = attention_block(dn_4, gating_5, filters[3])
    up_5 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(dn_5)
    up_5 = layers.concatenate([up_5, att_5], axis=3)
    up_conv_5 = res_conv_block(up_5, kernelsize, filters[3], dropout, batchnorm)
    
    gating_4 = gatingsignal(up_conv_5, filters[2], batchnorm)
    att_4 = attention_block(dn_3, gating_4, filters[2])
    up_4 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_5)
    up_4 = layers.concatenate([up_4, att_4], axis=3)
    up_conv_4 = res_conv_block(up_4, kernelsize, filters[2], dropout, batchnorm)
   
    gating_3 = gatingsignal(up_conv_4, filters[1], batchnorm)
    att_3 = attention_block(dn_2, gating_3, filters[1])
    up_3 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_4)
    up_3 = layers.concatenate([up_3, att_3], axis=3)
    up_conv_3 = res_conv_block(up_3, kernelsize, filters[1], dropout, batchnorm)
    
    gating_2 = gatingsignal(up_conv_3, filters[0], batchnorm)
    att_2 = attention_block(dn_1, gating_2, filters[0])
    up_2 = layers.UpSampling2D(size=(upsample_size, upsample_size), data_format="channels_last")(up_conv_3)
    up_2 = layers.concatenate([up_2, att_2], axis=3)
    up_conv_2 = res_conv_block(up_2, kernelsize, filters[0], dropout, batchnorm)
   
    conv_final = layers.Conv2D(1, kernel_size=(1,1))(up_conv_2)
    conv_final = layers.BatchNormalization(axis=3)(conv_final)
    outputs = layers.Activation('sigmoid')(conv_final)  

    model = models.Model(inputs=[inputs], outputs=[outputs])
    model.compile(optimizer = Adam(learning_rate = 1e-4), loss = 'binary_crossentropy', metrics = [dice_coef, mean_iou, Recall(), Precision(), 'accuracy'])


    return model

In [24]:

from tensorflow.keras import models, layers, regularizers
from tensorflow.keras import backend as K

train_batch = 3
valid_batch = 3
train_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/aug_train'
valid_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test'
#init_model_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/model/pretrained'
#save_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/model/new_VSEGmodel'
csv_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/model/CSV/data.csv'
num_train = 902
num_valid = 21
img_train_fpath = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/aug_train/final_img/'
img_valid_fpath = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test/final_img'


train_generator = trainGenerator(train_batch, train_path, 'image', 'mask', data_gen_args, save_to_dir = img_train_fpath, target_size=(512,512))
valid_generator = validGenerator(valid_batch, valid_path, 'image', 'mask', data_gen_args, save_to_dir = img_valid_fpath, target_size=(512,512))

input_shape = (512,512,1)
#model = AttentionUNet(input_shape, NUM_CLASSES=1, dropout_rate=0.2, batch_norm=True)
#  if initial_model_path != None:
#    model.load_weights(initial_model_path)

model = attentionunet(input_shape)
#  if initial_model_path != None:
model.load_weights('retina_attentionUnet_150epochs.hdf5')

model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 512, 512, 1  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_39 (Conv2D)             (None, 512, 512, 16  160         ['input_2[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization_27 (BatchN  (None, 512, 512, 16  64         ['conv2d_39[0][0]']              
 ormalization)                  )                                                           

                                                                                                  
 dropout_12 (Dropout)           (None, 64, 64, 128)  0           ['activation_37[0][0]']          
                                                                                                  
 conv2d_46 (Conv2D)             (None, 64, 64, 128)  147584      ['dropout_12[0][0]']             
                                                                                                  
 batch_normalization_34 (BatchN  (None, 64, 64, 128)  512        ['conv2d_46[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_38 (Activation)     (None, 64, 64, 128)  0           ['batch_normalization_34[0][0]'] 
                                                                                                  
 max_pooli

                                                                                                  
 conv2d_56 (Conv2D)             (None, 64, 64, 64)   8256        ['activation_45[0][0]']          
                                                                                                  
 batch_normalization_41 (BatchN  (None, 64, 64, 64)  256         ['conv2d_56[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 activation_46 (Activation)     (None, 64, 64, 64)   0           ['batch_normalization_41[0][0]'] 
                                                                                                  
 conv2d_58 (Conv2D)             (None, 64, 64, 64)   4160        ['activation_46[0][0]']          
                                                                                                  
 conv2d_tr

                                                                                                  
 add_6 (Add)                    (None, 128, 128, 32  0           ['conv2d_transpose_6[0][0]',     
                                )                                 'conv2d_64[0][0]']              
                                                                                                  
 activation_52 (Activation)     (None, 128, 128, 32  0           ['add_6[0][0]']                  
                                )                                                                 
                                                                                                  
 conv2d_66 (Conv2D)             (None, 128, 128, 1)  33          ['activation_52[0][0]']          
                                                                                                  
 activation_53 (Activation)     (None, 128, 128, 1)  0           ['conv2d_66[0][0]']              
          

                                                                                                  
 lambda_7 (Lambda)              (None, 512, 512, 16  0           ['up_sampling2d_14[0][0]']       
                                )                                                                 
                                                                                                  
 multiply_7 (Multiply)          (None, 512, 512, 16  0           ['lambda_7[0][0]',               
                                )                                 'activation_32[0][0]']          
                                                                                                  
 conv2d_74 (Conv2D)             (None, 512, 512, 16  272         ['multiply_7[0][0]']             
                                )                                                                 
                                                                                                  
 up_sampli

 batch_normalization_29 (BatchN  (None, 256, 256, 32  128        ['conv2d_41[0][0]']              
 ormalization)                  )                                                                 
                                                                                                  
 activation_33 (Activation)     (None, 256, 256, 32  0           ['batch_normalization_29[0][0]'] 
                                )                                                                 
                                                                                                  
 dropout_10 (Dropout)           (None, 256, 256, 32  0           ['activation_33[0][0]']          
                                )                                                                 
                                                                                                  
 conv2d_42 (Conv2D)             (None, 256, 256, 32  9248        ['dropout_10[0][0]']             
          

 activation_41 (Activation)     (None, 32, 32, 128)  0           ['batch_normalization_37[0][0]'] 
                                                                                                  
 conv2d_51 (Conv2D)             (None, 32, 32, 128)  16512       ['activation_41[0][0]']          
                                                                                                  
 conv2d_transpose_4 (Conv2DTran  (None, 32, 32, 128)  147584     ['conv2d_51[0][0]']              
 spose)                                                                                           
                                                                                                  
 conv2d_50 (Conv2D)             (None, 32, 32, 128)  65664       ['activation_38[0][0]']          
                                                                                                  
 add_4 (Add)                    (None, 32, 32, 128)  0           ['conv2d_transpose_4[0][0]',     
          

 conv2d_60 (Conv2D)             (None, 128, 128, 64  4160        ['multiply_5[0][0]']             
                                )                                                                 
                                                                                                  
 up_sampling2d_11 (UpSampling2D  (None, 128, 128, 12  0          ['activation_45[0][0]']          
 )                              8)                                                                
                                                                                                  
 batch_normalization_42 (BatchN  (None, 128, 128, 64  256        ['conv2d_60[0][0]']              
 ormalization)                  )                                                                 
                                                                                                  
 concatenate_5 (Concatenate)    (None, 128, 128, 19  0           ['up_sampling2d_11[0][0]',       
          

                                )                                                                 
                                                                                                  
 batch_normalization_47 (BatchN  (None, 256, 256, 32  128        ['conv2d_68[0][0]']              
 ormalization)                  )                                                                 
                                                                                                  
 activation_54 (Activation)     (None, 256, 256, 32  0           ['batch_normalization_47[0][0]'] 
                                )                                                                 
                                                                                                  
 dropout_16 (Dropout)           (None, 256, 256, 32  0           ['activation_54[0][0]']          
                                )                                                                 
          

                                                                                                  
 batch_normalization_52 (BatchN  (None, 512, 512, 16  64         ['conv2d_76[0][0]']              
 ormalization)                  )                                                                 
                                                                                                  
 activation_60 (Activation)     (None, 512, 512, 16  0           ['batch_normalization_52[0][0]'] 
                                )                                                                 
                                                                                                  
 conv2d_77 (Conv2D)             (None, 512, 512, 1)  17          ['activation_60[0][0]']          
                                                                                                  
 batch_normalization_53 (BatchN  (None, 512, 512, 1)  4          ['conv2d_77[0][0]']              
 ormalizat

In [180]:
callbacks = [
    #ModelCheckpoint(save_path, verbose=1, save_best_only=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=5, min_lr=1e-6, verbose=1),
    CSVLogger(csv_path),
    TensorBoard(),
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=False)
    ]

model_checkpoint = tf.keras.callbacks.ModelCheckpoint('AttResUNET2.hdf5', monitor='loss',verbose=1, save_best_only=True)

model.fit(train_generator,steps_per_epoch=num_train//train_batch,epochs=80,verbose=1,callbacks=[callbacks,model_checkpoint],
          validation_data=valid_generator, validation_steps=num_valid//valid_batch)

Found 902 images belonging to 1 classes.
Found 902 images belonging to 1 classes.
Epoch 1/80


2023-02-03 18:52:11.062328: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


300/300 [==============================] - ETA: 0s - loss: 0.6197 - dice_coef: 0.2143 - mean_iou: 0.1202 - recall_22: 0.8161 - precision_22: 0.2725 - accuracy: 0.8105Found 21 images belonging to 1 classes.
Found 21 images belonging to 1 classes.


2023-02-03 18:57:51.360333: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.



Epoch 1: loss improved from inf to 0.61970, saving model to AttResUNET2.hdf5
300/300 [==============================] - 347s 1s/step - loss: 0.6197 - dice_coef: 0.2143 - mean_iou: 0.1202 - recall_22: 0.8161 - precision_22: 0.2725 - accuracy: 0.8105 - val_loss: 0.5577 - val_dice_coef: 0.1987 - val_mean_iou: 0.1104 - val_recall_22: 0.6465 - val_precision_22: 0.5052 - val_accuracy: 0.9214 - lr: 1.0000e-04
Epoch 2/80
 27/300 [=>............................] - ETA: 5:08 - loss: 0.5813 - dice_coef: 0.2243 - mean_iou: 0.1265 - recall_22: 0.8486 - precision_22: 0.4066 - accuracy: 0.8925

KeyboardInterrupt: 

In [174]:
import sklearn.metrics as sm

def get_confusion_matrix_elements(groundtruth_list, predicted_list):
    """returns confusion matrix elements i.e TN, FP, FN, TP as floats
	See example code for helper function definitions
    """
    tn, fp, fn, tp = sm.confusion_matrix(groundtruth_list, predicted_list,labels=[0,1]).ravel()
    tn, fp, fn, tp = np.float64(tn), np.float64(fp), np.float64(fn), np.float64(tp)

    return tn, fp, fn, tp

def get_prec_rec_IoU_accuracy(groundtruth_list, predicted_list):
    """returns precision, recall, IoU and accuracy metrics
	"""
    tn, fp, fn, tp = get_confusion_matrix_elements(groundtruth_list, predicted_list)
    
    total = tp + fp + fn + tn
    accuracy = (tp + tn) / total
    prec=tp/(tp+fp)
    rec=tp/(tp+fn)
    IoU=tp/(tp+fp+fn)
    
    return prec,rec,IoU,accuracy

def get_f1_score(groundtruth_list, predicted_list):
    """Return f1 score covering edge cases"""

    tn, fp, fn, tp = get_confusion_matrix_elements(groundtruth_list, predicted_list)
    
    f1_score = (2 * tp) / ((2 * tp) + fp + fn)

    return f1_score

def get_validation_metrics(groundtruth,predicted):
    """Return all output metrics. Input is binary images"""
   
    u,v=np.shape(groundtruth)
    groundtruth_list=np.reshape(groundtruth,(u*v,))
    predicted_list=np.reshape(predicted,(u*v,))
    prec,rec,IoU,acc=get_prec_rec_IoU_accuracy(groundtruth_list, predicted_list)
    f1_score=get_f1_score(groundtruth_list, predicted_list)
   # print("Precision=",prec, "Recall=",rec, "IoU=",IoU, "acc=",acc, "F1=",f1_score)
    return prec,rec,IoU,acc,f1_score

def evalResult(gth_path,npyfile,target_size=(512,512),flag_multi_class = False,num_class = 2):
    files=sorted(os.listdir(gth_path))
    print(files)
    prec=0
    rec=0
    acc=0
    IoU=0
    f1_score=0
    for i,item in enumerate(npyfile):
        img = item[:,:,0]
        gth = io.imread(os.path.join(gth_path,files[i]))
        gth = trans.resize(gth,target_size)
        img1=np.array(((img - np.min(img))/np.ptp(img))>0.1).astype(float)
        gth1=np.array(((gth - np.min(gth))/np.ptp(gth))>0.1).astype(float)
        p,r,I,a,f=get_validation_metrics(gth1,img1)
        prec=prec+p
        rec=rec+r
        acc=acc+a
        IoU=IoU+I
        f1_score=f1_score+f
    print("Precision=",prec/(i+1), "Recall=",rec/(i+1), "IoU=",IoU/(i+1), "acc=",acc/(i+1), "F1=",f1_score/(i+1))    

In [175]:
test_img_path = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test/image/'
n_i=len(os.listdir(test_img_path))
test_gen = testGenerator(test_img_path)
results = model.predict_generator(test_gen,n_i,verbose=1)
save_result = '/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test/predicted_labels/'
saveResult(test_img_path,save_result,results)

/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2686884916.py:4: UserWarning: `Model.predict_generator` is deprecated and will be removed in a future version. Please use `Model.predict`, which supports generators.
  results = model.predict_generator(test_gen,n_i,verbose=1)


13.jpg


2023-02-03 18:29:08.569546: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


14.jpg
21/21 [==============================] - 12s 461ms/step


In [176]:
gt_path='/Users/sadda/Documents/Science_Fair/Datasets/RSEG/new_data/test/mask/'
evalResult(gt_path,results)

['13.jpg', '14.jpg', '16.jpg', '19.jpg', '25.jpg', '29.jpg', '33.jpg', '39.jpg', '40.jpg', '48.jpg', '55.jpg', '6.jpg', '61.jpg', '64.jpg', '67.jpg', '73.jpg', '82.jpg', '85.jpg', '90.jpg', '94.jpg', '97.jpg']


/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:57: RuntimeWarning: invalid value encountered in divide
  img1=np.array(((img - np.min(img))/np.ptp(img))>0.1).astype(float)
/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:19: RuntimeWarning: invalid value encountered in scalar divide
  prec=tp/(tp+fp)
/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:57: RuntimeWarning: invalid value encountered in divide
  img1=np.array(((img - np.min(img))/np.ptp(img))>0.1).astype(float)
/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:19: RuntimeWarning: invalid value encountered in scalar divide
  prec=tp/(tp+fp)
/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:57: RuntimeWarning: invalid value encountered in divide
  img1=np.array(((img - np.min(img))/np.ptp(img))>0.1).astype(float)
/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/245

Precision= nan Recall= 0.0 IoU= 0.0 acc= 0.9004981631324405 F1= 0.0


/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:57: RuntimeWarning: invalid value encountered in divide
  img1=np.array(((img - np.min(img))/np.ptp(img))>0.1).astype(float)
/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:19: RuntimeWarning: invalid value encountered in scalar divide
  prec=tp/(tp+fp)
/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:57: RuntimeWarning: invalid value encountered in divide
  img1=np.array(((img - np.min(img))/np.ptp(img))>0.1).astype(float)
/var/folders/jm/m054n7_s0bn2215whpnw82r06kh0mk/T/ipykernel_96146/2456749743.py:19: RuntimeWarning: invalid value encountered in scalar divide
  prec=tp/(tp+fp)
